# Data Preparation for Task Durations
This notebook is responsible for parsing the raw BPIC-17 event log and extracting the active working durations for human-driven tasks (`W_` activities). 

### Imports & Setup

In [2]:
import pm4py as pm
import pandas as pd

In [3]:
log = pm.read_xes('../../data/BPI Challenge 2017.xes.gz')
df_raw = pm.convert_to_dataframe(log)

/Users/matildabriegel/opt/anaconda3/envs/bppso/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [4]:
df_whole = df_raw.copy()
df_whole.rename(columns={
    'org:resource': 'Resource',
    'concept:name': 'Activity',
    'time:timestamp': 'Timestamp',
    'lifecycle:transition': 'State',
    'case:concept:name': 'Case'
}, inplace=True)
df_whole.drop(columns=['EventOrigin', 'EventID', 'OfferID'], inplace=True)  # TODO: Check if these are needed later
df_whole = df_whole.sort_values(by=['Case', 'Activity', 'Timestamp'])

In [5]:
df = df_whole[~df_whole['Activity'].isin(['W_Assess potential fraud', 'W_Shortened completion ', 'W_Personal Loan collection', 'O_Sent (online only)'])]     # Activities not in bpmn model

### Define data

Define which columns to use for training

In [6]:
case_cols = ['case:LoanGoal', 'case:RequestedAmount', 'case:ApplicationType']
#business_cols = ['CreditScore', 'OfferedAmount', 'MonthlyCost', 'NumberOfTerms', 'FirstWithdrawalAmount', 'Accepted', 'Selected']
context = case_cols #+ business_cols

In [7]:
df[context] = df.groupby('Case')[context].ffill()

/var/folders/rz/mjmkgf7967x_k_v3fxtyf9th0000gn/T/ipykernel_33401/2393894474.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[context] = df.groupby('Case')[context].ffill()


In [8]:
starts = df[df['State'] == 'start'][['Case', 'Activity', 'Resource', 'Timestamp']]
completes = df[df['State'] == 'complete'][['Case', 'Activity', 'Resource', 'Timestamp'] + context]

In [9]:
nn_data = pd.merge(
    starts, 
    completes, 
    on=['Case', 'Activity', 'Resource'], 
    suffixes=('_start', '_end')
)

nn_data['Duration'] = (nn_data['Timestamp_end'] - nn_data['Timestamp_start']).dt.total_seconds()

In [10]:
final_columns = ['Resource', 'Activity'] + context + ['Duration']
final_csv = nn_data[final_columns].copy()

final_csv = final_csv[final_csv['Duration'] >= 0]

#final_csv[business_cols] = final_csv[business_cols].fillna(-1)  # Start of a process instance, not yet defined
final_csv.to_csv('nn_data_case_context.csv', index=False)

Calculate average times for resources per activity as baseline model

In [15]:
nn_data = nn_data[nn_data['Duration'] >= 0] # Filter out noise
resource_activity_avgs = nn_data.groupby(['Resource', 'Activity'])['Duration'].mean().reset_index()
resource_activity_avgs.rename(columns={'Duration': 'AverageDuration'}, inplace=True)

resource_activity_avgs.to_csv('resource_activity_averages.csv', index=False)
print("Created 'resource_activity_averages.csv'.")


Created 'resource_activity_averages.csv'.
